# Water usage analysis
Run all cells from the repository root or src/. The processed water_data.csv is included.
This notebook retains the historical analysis; it is not a fresh validation of all manuscript summaries.
The current forest plot is generated by `python3 src/realdata_figure.py` from the rounded manuscript table.
Current notebook result: CEB approximately −0.37 [−0.80, 0.06]; manuscript/forest plot: −0.58 [−0.90, −0.27]. Reconciliation remains pending; see README.


In [ ]:
import logging
import pandas as pd
from scipy.stats import norm
from scipy.optimize import minimize_scalar
import matplotlib.pyplot as plt
import numpy as np

Data Description: the experimental subjects were “residential customers who lived in their homes from May 2006 to April 2007 and used at least 20,000 gallons during the 2006 summer watering season (about 80 percent of the population).” (p.363) Mailings that encouraged water conservation were sent to approximately 11,700 households, with roughly 71,800 households serving as controls.[1]  Outcomes were measured using the utility company’s records of water usage at each address.

Note that the main focus is on the control group treatment==4 and the social norms treatment treatment==3.

# Load the processed data

In [ ]:
from pathlib import Path
DATA_DIR = Path("src") if Path("src/water_data.csv").exists() else Path(".")
water_df = pd.read_csv(DATA_DIR / "water_data.csv")

In [ ]:
rng = np.random.default_rng(42)

Focus on treatment 3 and control group

In [ ]:
filtered_df = water_df.loc[
    (water_df['treat3'] == 1) |
    (water_df['treat1'] == 0) &
    (water_df['treat2'] == 0) &
    (water_df['treat3'] == 0), :
]
filtered_df = filtered_df.drop(
    columns=['treat1', 'treat2','summer2007', 'summer2009'] +
    [col for col in filtered_df.columns if 't1' in col or 't2' in col or "t3_" in col]
).rename(columns={'treat3': 'treat'})

In [ ]:
filtered_df.columns

## Check missing values

Check missing values and drop any rows with missing data

In [ ]:
filtered_df.isnull().mean()

In [ ]:
filtered_df = filtered_df.dropna()

In [ ]:
filtered_df.columns

## Compute ATE with the full data

In [ ]:
y1 = filtered_df.loc[filtered_df.treat == 1, "summer2008"].to_numpy(float)
y0 = filtered_df.loc[filtered_df.treat == 0, "summer2008"].to_numpy(float)

# Point estimate
theta = y1.mean() - y0.mean()

# Plug-in variance
var_theta = y1.var(ddof=1) / y1.size + y0.var(ddof=1) / y0.size
se_theta = np.sqrt(var_theta)

# 95% CI (Normal approximation)
lower_bound = theta - 1.96 * se_theta
upper_bound = theta + 1.96 * se_theta

print(f"Point estimate of θ: {theta:.4f}")
print(f"Variance: {var_theta:.6f}")
print(f"95% Confidence Interval: [{lower_bound:.4f}, {upper_bound:.4f}]")

## Constructing Experimetnal, Observational and Calibration Estimators

Split the data into 100 equally sized disjoint subsets by stratified sampling: sample the treatment and control groups separately and then combining them.

In [ ]:
num_splits = 100

df_treat = filtered_df.loc[filtered_df["treat"] == 1].sample(frac=1, random_state=42).reset_index(drop=True)
df_ctrl  = filtered_df.loc[filtered_df["treat"] == 0].sample(frac=1, random_state=42).reset_index(drop=True)

t_idx = np.array_split(np.arange(len(df_treat)), num_splits)
c_idx = np.array_split(np.arange(len(df_ctrl)),  num_splits)

subsets = [
    pd.concat([df_treat.iloc[t_idx[i]], df_ctrl.iloc[c_idx[i]]], ignore_index=True)
      .sample(frac=1, random_state=42)
      .reset_index(drop=True)
    for i in range(num_splits)
]

### Experimental Design

Assign the first subset as experimental data. Since treatment is randomized, we combine these subsets and compute the ATE and its variance on the pooled sample.

In [ ]:
experimental_subsets = subsets[:1]
df_e = pd.concat(experimental_subsets, ignore_index=True)

o1 = df_e.loc[df_e.treat == 1, "summer2008"].to_numpy(float)
o0 = df_e.loc[df_e.treat == 0, "summer2008"].to_numpy(float)

y_e = o1.mean() - o0.mean()
sigma2_e = o1.var(ddof=1) / o1.size + o0.var(ddof=1) / o0.size

print(f"ATE: {y_e:.6f}")
print(f"Var(ATE): {sigma2_e:.6f}")

### Observational Design

In [ ]:
observational_subsets = subsets[1:]

Define a new propensity score model in which the pre-treatment water use in summer 2006 has a strong positive effect on receiving the treatment.

In [ ]:
def e_obs(X, beta):
    z = X @ beta
    z = (z - z.mean()) / (z.std() + 1e-12)
    return 1.0 / (1.0 + np.exp(-z))

covariates = ["summer2006"]
beta = np.array([0.5])

In [ ]:
y_obs, sigma2_obs = [], []

for df in observational_subsets:
    X = df[covariates].to_numpy(float)
    p = e_obs(X, beta)

    a = df["treat"].to_numpy(int)
    w = a / np.clip(p, 1e-6, 1 - 1e-6) + (1 - a) / np.clip(1 - p, 1e-6, 1 - 1e-6)
    prob = w / w.sum()

    idx = rng.choice(len(df), size=len(df), replace=True, p=prob)
    df_obs = df.iloc[idx]

    o1 = df_obs.loc[df_obs.treat == 1, "summer2008"].to_numpy(float)
    o0 = df_obs.loc[df_obs.treat == 0, "summer2008"].to_numpy(float)

    y = o1.mean() - o0.mean()
    v = o1.var(ddof=1) / o1.size + o0.var(ddof=1) / o0.size

    y_obs.append(y)
    sigma2_obs.append(v)

y_obs = np.array(y_obs)
sigma2_obs = np.array(sigma2_obs)

print("Observational mean bias is", np.mean(y_obs) - theta)

Apply the illusion model to combine the experimental the observational estimates.
1) Estimate $\sigma^2_b$ by maximizing the marginal likelihood (given in the proof of Theorem 1).

In [ ]:
def obj_sigma_b2(sigma_b2, y_obs, sigma2_obs, sigma2_e):
    d = sigma2_obs + sigma_b2
    w = 1.0 / d
    y_tilde = (w @ y_obs) / w.sum()
    term1 = -np.sum((y_obs - y_tilde) ** 2 / d)
    term2 = -np.sum(np.log(d))
    term3 = -np.log(1.0 / sigma2_e + np.sum(w))
    return term1 + term2 + term3

def fit_mmle(y_obs, sigma2_obs, sigma2_e, upper=1e6):
    y_obs = np.asarray(y_obs, dtype=float)
    sigma2_obs = np.asarray(sigma2_obs, dtype=float)
    eps = 1e-12
    sigma2_obs = np.maximum(sigma2_obs, eps)

    res = minimize_scalar(
        lambda s: -obj_sigma_b2(s, y_obs, sigma2_obs, sigma2_e),
        bounds=(0.0, upper),
        method="bounded",
        options={"xatol": 1e-10},
    )
    return res.x, obj_sigma_b2(res.x, y_obs, sigma2_obs, sigma2_e), res

sigma2_b, obj_hat, res = fit_mmle(y_obs, sigma2_obs, sigma2_e)

2) Compute the illusion posterior mean and variance using the estimated $\sigma^2_b$.

In [ ]:
illusion_mean = y_e
illusion_var = 1.0 / (
    1.0 / sigma2_e + np.sum(1.0 / (sigma2_obs + sigma2_b))
)

### Calibration Design

In [ ]:
calibration_subsets = subsets[1:]

In [ ]:
y_n, sigma2_n = [], []

for df in calibration_subsets:
    X = df[covariates].to_numpy(float)
    p = np.clip(e_obs(X, beta), 1e-6, 1 - 1e-6)

    a_tilde = rng.binomial(1, p, size=len(df))

    o = df["summer2008"].to_numpy(float)
    o1 = o[a_tilde == 1]
    o0 = o[a_tilde == 0]

    y = o1.mean() - o0.mean()
    v = o1.var(ddof=1) / o1.size + o0.var(ddof=1) / o0.size

    y_n.append(y)
    sigma2_n.append(v)

y_n = np.array(y_n)
sigma2_n = np.array(sigma2_n)

print("Calibration mean:", np.mean(y_n))
print("Calibration mean variance:", np.mean(sigma2_n))

Apply the calibrated illusion model to combine the experimental, observational and calibration estimates. We estimate $\mu_b$ and $\sigma^2_b$ with moment estimators.

In [ ]:
mu_b = np.mean(y_n)
sigma2_b = np.sum((y_n - mu_b)**2) / (len(y_n) - 1)

# Precision terms
prec_e = 1.0 / sigma2_e
prec_obs = 1.0 / (sigma2_obs + sigma2_b)

# Empirical Bayes posterior mean
EB_post_mean = (
    prec_e * y_e
    + np.sum(prec_obs * (y_obs + mu_b))
) / (
    prec_e + np.sum(prec_obs)
)

# Empirical Bayes posterior variance
EB_post_var = 1.0 / (
    prec_e + np.sum(prec_obs)
)

print(f"EB posterior mean: {EB_post_mean:.6f}")
print(f"EB posterior variance: {EB_post_var:.6f}")

In [ ]:
z = 1.96

# Reduced experimental posterior (Exp1) — baseline
exp_mean = y_e
exp_var  = sigma2_e
exp_sd   = np.sqrt(exp_var)
exp_lo   = exp_mean - z * exp_sd
exp_hi   = exp_mean + z * exp_sd

# Posterior expected MSE (baseline)
err_exp1 = exp_var + (exp_mean - theta) ** 2
exp_p_in_trueCI = (
    norm.cdf(upper_bound, loc=exp_mean, scale=exp_sd)
    - norm.cdf(lower_bound, loc=exp_mean, scale=exp_sd)
)

# Illusion posterior
ill_mean = illusion_mean
ill_var  = illusion_var
ill_sd   = np.sqrt(ill_var)
ill_lo   = ill_mean - z * ill_sd
ill_hi   = ill_mean + z * ill_sd
ill_p_in_trueCI = (
    norm.cdf(upper_bound, loc=ill_mean, scale=ill_sd)
    - norm.cdf(lower_bound, loc=ill_mean, scale=ill_sd)
)

err_ill = ill_var + (ill_mean - theta) ** 2
ill_rel_prec = err_exp1 / err_ill


# No-illusion (EB calibrated) posterior
EB_mean = EB_post_mean
EB_var  = EB_post_var
EB_sd   = np.sqrt(EB_var)
EB_lo   = EB_mean - z * EB_sd
EB_hi   = EB_mean + z * EB_sd
EB_p_in_trueCI = (
    norm.cdf(upper_bound, loc=EB_mean, scale=EB_sd)
    - norm.cdf(lower_bound, loc=EB_mean, scale=EB_sd)
)

err_EB = EB_var + (EB_mean - theta) ** 2
EB_rel_prec = err_exp1 / err_EB


# Full experimental ATE (frequentist CI)
full_mean = theta
full_lo   = lower_bound
full_hi   = upper_bound

# Report (rounded to 2 decimals)
print("=== Experimental (reduced) ===")
print(f"Mean: {exp_mean:.2f}")
print(f"95% CI: [{exp_lo:.2f}, {exp_hi:.2f}]")
print(f"P(theta in true CI): {exp_p_in_trueCI:.2f}")
print(f"Relative precision: {1:.2f}\n")

print("=== Illusion posterior (exp + obs, no calibration) ===")
print(f"Mean: {ill_mean:.2f}")
print(f"95% CI: [{ill_lo:.2f}, {ill_hi:.2f}]")
print(f"P(theta in true CI): {ill_p_in_trueCI:.4f}")
print(f"Relative precision: {ill_rel_prec:.2f}\n")

print("=== No-illusion posterior (EB with calibration) ===")
print(f"Mean: {EB_mean:.2f}")
print(f"95% CI: [{EB_lo:.2f}, {EB_hi:.2f}]")
print(f"P(theta in true CI): {EB_p_in_trueCI:.2f}")
print(f"Relative precision: {EB_rel_prec:.2f}\n")

print("=== Experiment (full sample) ===")
print(f"Mean: {full_mean:.2f}")
print(f"95% CI: [{full_lo:.2f}, {full_hi:.2f}]")
print("P(theta > 0): NA")
print(f"Relative precision: {err_exp1/var_theta:.2f}\n")

## Manuscript figure
Run `python3 src/realdata_figure.py` from the repository root to generate `figures/sim_realdata.pdf`. The plot uses the table summaries; it does not rerun this notebook.